In [ ]:
import os
import pandas as pd
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from dotenv import load_dotenv
load_dotenv()

from google import genai
key = os.getenv("ai_key")

client = genai.Client(api_key=key)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stackoverflow/pythonquestions")

print("Path to dataset files:", path)

In [ ]:
questions = pd.read_csv(os.path.join(path, "Questions.csv"), encoding="latin1")
answers   = pd.read_csv(os.path.join(path, "Answers.csv"), encoding="latin1")
tags      = pd.read_csv(os.path.join(path, "Tags.csv"), encoding="latin1")

In [ ]:
questions.head()

In [ ]:
questions.info()
answers.info()
tags.info()

In [ ]:
questions.describe()

In [ ]:
questions = questions[questions["Score"] > 0]

In [ ]:
answers.describe()

In [ ]:
answers = answers[answers["Score"] > 0].copy()

In [ ]:
# 1. Compute the mean score for each question
mean_scores = answers.groupby("ParentId")["Score"].mean().reset_index()
mean_scores.rename(columns={"Score": "mean_score"}, inplace=True)

# 2. Attach the mean score to each answer
answers_with_mean = answers.merge(
    mean_scores,
    on="ParentId",
    how="left"
)

# 3. Keep only answers with score > mean score of that question
answers_filtered = answers_with_mean[
    answers_with_mean["Score"] > answers_with_mean["mean_score"]
]

# 4. Group only the filtered answers
#answers_grouped = answers_filtered.groupby("ParentId")["Body"].apply(list).reset_index()
answers_grouped = (
    answers_filtered
    .groupby("ParentId")
    .apply(
        lambda g: [
            {
                "body": row["Body"],
                "score": row["Score"]
            }
            for _, row in g.iterrows()
        ]
    )
    .reset_index(name="answers_list")
)

answers_grouped.head()

In [ ]:
# 5. Merge questions with the filtered answers
df = questions.merge(
    answers_grouped,
    left_on="Id",
    right_on="ParentId",
    how="left"
)

# 6. Replace NaN with empty lists for questions without good answers
df["answers_list"] = df["answers_list"].apply(lambda x: x if isinstance(x, list) else [])

# 7. Rename columns for clarity
df.rename(columns={
    "Body_x": "question_body",
}, inplace=True)

In [ ]:
df.head()

In [ ]:
df = df[["Id", "Title", "Body","Score", "answers_list"]]
df.head()

In [ ]:
def build_document(row):
    question_score = row.get("Score", "")
    answers = row.get("answers_list") or []

    formatted_answers = []
    for i, ans in enumerate(answers, start=1):
        formatted_answers.append(
            f"Answer {i} - Score: {ans['score']}\n{ans['body']}"
        )

    answers_text = "\n\n".join(formatted_answers)

    return (
        f"Title: {row.get('Title', '')}\n"
        f"Question Score: {question_score}\n\n"
        f"Question:\n{row.get('Body', '')}\n\n"
        f"Answers:\n{answers_text}"
    )


df["document"] = df.apply(build_document, axis=1)

In [ ]:
print(df.document.iloc[50])

In [ ]:
def clean(text):
    return BeautifulSoup(str(text), "html.parser").get_text()

df["document"] = df["document"].apply(clean)


In [ ]:
print(df.document.iloc[0])

In [ ]:
def fix_surrogates(text):
    if isinstance(text, str):
        return text.encode("utf-8", "surrogatepass").decode("utf-8", "ignore")
    return text

df = df.applymap(fix_surrogates)

In [ ]:
# Crear rutas
df_clean_path =  "df_clean.parquet"


# Guardar
df.to_parquet(df_clean_path)


# Leer
df = pd.read_parquet(df_clean_path)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")


documents = list(df["document"].values)

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=512,
    normalize_embeddings=True
).astype("float32")

dim = embeddings.shape[1]


In [ ]:
"""
model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en")

documents = df["document"].tolist()

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32
).astype("float32")

dim = embeddings.shape[1]
print(dim)
"""


In [ ]:
index = faiss.IndexFlatL2(dim)
index.add(embeddings)


In [ ]:
def retrieve(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    distances, indices = index.search(q_emb, k)
    return df.loc[indices[0], ["document"]]

In [ ]:
docs = retrieve("How can I open a CSV file in Python?")

In [ ]:
for i, doc in enumerate(docs["document"]):
    print(f"\n===== DOCUMENT {i} =====\n")
    print(doc)

In [ ]:
def build_context(rows):
    return "\n\n---\n\n".join(rows["document"].tolist())


In [ ]:
def call_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

In [ ]:
def rag_answer(query):
    retrieved = retrieve(query)

    # Construimos el contexto a partir de los documentos ya formateados
    context = build_context(retrieved)

    prompt = f"""
    You are an expert Python assistant. Answer ONLY using the information in the context.

    ---------------- CONTEXT START ----------------
    {context}
    ---------------- CONTEXT END ------------------

    QUESTION:
    {query}

    FINAL ANSWER (in English):
    """

    # Llamada al modelo
    answer = call_gemini(prompt)

    # Devolvemos ambos
    return {
        "prompt": prompt,
        "retrieved_docs": retrieved,
        "answer": answer
    }

In [ ]:
result = rag_answer("How do I open a CSV in Python?")


print(result["answer"])

In [ ]:
print(result["prompt"])

In [ ]:
result = rag_answer("¿How can I open a CSV file in Python using Pandas?")

In [ ]:
print(result["answer"])

In [ ]:
result = rag_answer("Give me an example of Inheritance in Python")
print(result["answer"])

In [ ]:
result = rag_answer("what are the best libraries for Machine Learning in Python")
print(result["answer"])

In [ ]:
result = rag_answer("is Python a snake?")
print(result["answer"])